# Cloud Resource and Cost Optimization Advisor — Colab Validation

**BITS PGAIML · PCAM ZG519 Agentic AI · Assignment-1 · Group 11**  
Repo: https://github.com/bits-pgaiml-aai/pcam-zg519-a1-cloud-advisor

This notebook:
1. Clones / installs the project
2. Runs all **five classical agents** on shared scenarios A–D
3. Validates expected decision patterns
4. Exercises **mock LLM tools** (≥2 tools) and the LLM agent path
5. Demonstrates **Learning agent** experience updates

> **Note:** Full local Ollama (`qwen3:latest`) is optional here. Colab usually cannot run Ollama; the LLM path uses tool-evidence grounding / fallback so validation still passes.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:" , IN_COLAB)

REPO_URL = "https://github.com/bits-pgaiml-aai/pcam-zg519-a1-cloud-advisor.git"
REPO_DIR = Path("/content/pcam-zg519-a1-cloud-advisor") if IN_COLAB else Path.cwd().resolve()

# If opened from repo/notebooks locally, project root is parent.
if not IN_COLAB:
    if (Path.cwd() / "src").exists():
        REPO_DIR = Path.cwd()
    elif (Path.cwd().parent / "src").exists():
        REPO_DIR = Path.cwd().parent

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
else:
    os.chdir(REPO_DIR)

print("Project root:", REPO_DIR)
assert (REPO_DIR / "src" / "runner.py").exists(), "src/runner.py not found — check clone/path"

%pip install -q pandas langchain-core langchain-ollama ollama

In [ ]:
from pathlib import Path
import json
from copy import deepcopy

import pandas as pd

from src.agents.learning import LearningAgent
from src.runner import run_all_agents, simulate_learning_feedback
from src.tools.cloud_tools import (
    get_cost_estimate,
    get_resource_metrics,
    get_scaling_policy,
    get_service_level_targets,
)
from src.world import CloudWorld

world = CloudWorld()
print("Scenarios:", [s["id"] for s in world.list_scenarios()])

## 2. Shared-scenario fairness check

All agents must receive the **same** percept for a selected scenario.

In [ ]:
def assert_same_percept(scenario_id: str) -> dict:
    p1 = world.perceive(scenario_id)
    p2 = world.perceive(scenario_id)
    assert p1["metrics"] == p2["metrics"], "Metrics differ across perceives"
    assert p1["sla"] == p2["sla"], "SLA differs across perceives"
    return p1

for sid in ["A", "B", "C", "D"]:
    percept = assert_same_percept(sid)
    print(f"[{sid}] OK — instance={percept['metrics']['instance_type']}, kind={percept['scenario_kind']}")

print("\nPASS: shared scenario percepts are stable")

## 3. Run classical agents on A–D

In [ ]:
WEIGHTS = {"cost": 0.40, "performance": 0.40, "availability": 0.20}
RUNS = {}

rows = []
for sid in ["A", "B", "C", "D"]:
    payload = run_all_agents(
        sid,
        include_llm=False,
        utility_weights=WEIGHTS,
    )
    RUNS[sid] = payload
    for row in payload["comparison"]:
        rows.append(
            {
                "scenario": sid,
                "kind": payload["scenario"]["kind"],
                "agent": row["agent"],
                "recommendation": row["recommendation"],
                "display": row.get("display"),
            }
        )

summary = pd.DataFrame(rows)
pivot = summary.pivot(index="agent", columns="scenario", values="recommendation")
try:
    from IPython.display import display
    display(pivot)
except Exception:
    print(pivot.to_string())
summary.to_json("notebooks_colab_classical_summary.json", orient="records", indent=2)
print("Saved notebooks_colab_classical_summary.json")

## 4. Results validation (classical)

Checks align with assignment scenarios:
- **A (critical):** high CPU / latency → prefer `scale_up`
- **B (normal/cost):** low util + expensive → prefer `scale_down` for reflex/utility-style agents
- **C (variable load):** large peak/off-peak gap → prefer `enable_autoscaling` for reflex agents
- **D (edge):** agents may diverge (allowed)
- Required fields present on every decision

In [ ]:
REQUIRED_FIELDS = [
    "agent",
    "recommendation",
    "reasoning",
    "current_state_summary",
    "trade_offs",
    "monitoring_plan",
]

CLASSICAL = [
    "Simple Reflex",
    "Model-Based Reflex",
    "Goal-Based",
    "Utility-Based",
    "Learning",
]

failures = []

def recs(sid: str) -> dict:
    return {r["agent"]: r["recommendation"] for r in RUNS[sid]["comparison"]}

# Field completeness
for sid, payload in RUNS.items():
    agents_seen = []
    for result in payload["results"]:
        agents_seen.append(result["agent"])
        for field in REQUIRED_FIELDS:
            if field not in result or result[field] in (None, ""):
                failures.append(f"{sid}/{result.get('agent')}: missing {field}")
    for name in CLASSICAL:
        if name not in agents_seen:
            failures.append(f"{sid}: missing agent {name}")

# Scenario A: critical → scale_up for at least reflex + goal/utility/learning family
a = recs("A")
for name in ["Simple Reflex", "Model-Based Reflex", "Goal-Based", "Utility-Based", "Learning"]:
    if a.get(name) != "scale_up":
        failures.append(f"A/{name}: expected scale_up, got {a.get(name)}")

# Scenario B: low util expensive → scale_down for Simple/Model/Utility/Learning
b = recs("B")
for name in ["Simple Reflex", "Model-Based Reflex", "Utility-Based", "Learning"]:
    if b.get(name) != "scale_down":
        failures.append(f"B/{name}: expected scale_down, got {b.get(name)}")
# Goal-Based may pick first feasible plan (autoscaling) — allow divergence
if b.get("Goal-Based") not in {"scale_down", "enable_autoscaling", "rightsize_compute"}:
    failures.append(f"B/Goal-Based: unexpected {b.get('Goal-Based')}")

# Scenario C: peak/off-peak → enable_autoscaling for reflex agents
c = recs("C")
for name in ["Simple Reflex", "Model-Based Reflex"]:
    if c.get(name) != "enable_autoscaling":
        failures.append(f"C/{name}: expected enable_autoscaling, got {c.get(name)}")

# Scenario D: edge — allow divergence, but must be a known action
allowed = {
    "scale_up",
    "scale_down",
    "enable_autoscaling",
    "rightsize_compute",
    "keep_current",
    "escalate_to_human",
}
d = recs("D")
for name, action in d.items():
    if action not in allowed:
        failures.append(f"D/{name}: unknown action {action}")

# Agents must not all produce identical labels on D (edge should show mechanism differences often)
unique_d = set(d.values())
print("Scenario D recommendations:", d)
print("Unique actions on D:", unique_d)

if failures:
    print("\nFAIL:")
    for f in failures:
        print(" -", f)
    raise AssertionError(f"{len(failures)} validation failure(s)")

print("\nPASS: classical agent validation")

## 5. Tool validation (LLM prerequisites)

Assignment requires ≥2 tools with evidence. Validate mock tools directly.

In [ ]:
tool_failures = []

metrics = get_resource_metrics("B")
if "metrics" not in metrics or metrics["scenario_id"] != "B":
    tool_failures.append("get_resource_metrics failed for B")

inst = metrics["metrics"]["instance_type"]
cost = get_cost_estimate(inst)
if "hourly_usd" not in cost:
    tool_failures.append("get_cost_estimate missing hourly_usd")

sla = get_service_level_targets()
if "sla_targets" not in sla:
    tool_failures.append("get_service_level_targets missing sla_targets")

policy = get_scaling_policy("C")
if "recommended_policy" not in policy:
    tool_failures.append("get_scaling_policy missing recommended_policy")

print(json.dumps({"metrics_B": metrics, "cost": cost, "sla": sla, "policy_C": policy}, indent=2)[:1200], "...")

if tool_failures:
    raise AssertionError(tool_failures)

print("\nPASS: all four mock tools returned structured evidence")

## 6. LLM agent path (Ollama optional)

Tries `qwen3:latest` if Ollama is reachable; otherwise uses the assignment-safe tool-evidence fallback and still validates ≥2 tools in the trace.

In [ ]:
os.environ.setdefault("OLLAMA_MODEL", "qwen3:latest")
os.environ.setdefault("OLLAMA_HOST", "http://localhost:11434")

ALLOWED_ACTIONS = {
    "scale_up",
    "scale_down",
    "enable_autoscaling",
    "rightsize_compute",
    "keep_current",
    "escalate_to_human",
}

llm_payload = run_all_agents(
    "A",
    include_llm=True,
    utility_weights=WEIGHTS,
    llm_model=os.environ["OLLAMA_MODEL"],
    llm_host=os.environ["OLLAMA_HOST"],
)

llm_rows = [r for r in llm_payload["results"] if str(r["agent"]).startswith("LLM")]
assert llm_rows, "LLM agent result missing"
llm = llm_rows[0]

trace = llm.get("tool_trace") or []
tools_called = [t.get("tool") for t in trace]
print("LLM agent:", llm["agent"])
print("Recommendation:", llm.get("display_recommendation") or llm["recommendation"])
print("Tools called:", tools_called)

assert len(tools_called) >= 2, f"Expected >=2 tools, got {tools_called}"
assert llm["recommendation"] in ALLOWED_ACTIONS, llm["recommendation"]
assert llm.get("reasoning"), "LLM reasoning empty"

print("\nPASS: LLM path returned grounded recommendation with >=2 tool calls")


## 7. Learning agent experience update

In [ ]:
learner = LearningAgent(explore=0.0, seed=0)
before = deepcopy(learner.history["action_stats"]["scale_up"])

update = simulate_learning_feedback(
    learner,
    scenario_id="A",
    action="scale_up",
    success=True,
    notes="Colab validation success for scale_up",
)
after = learner.history["action_stats"]["scale_up"]

print("Before:", before)
print("Update:", update)
print("After:", after)

assert after["success"] == before["success"] + 1, "Learning success count did not increment"
assert learner.success_rate("scale_up") > 0.5, "Expected success_rate to rise above prior 0.5"

decision = learner.decide(world.perceive("A"))
assert "learning_stats" in decision
assert "limitations" in decision and decision["limitations"]
print("Learning decision:", decision["recommendation"])
print("\nPASS: learning history update affects reported stats")

## 8. Final validation scorecard

In [ ]:
scorecard = pd.DataFrame(
    [
        {"check": "Shared scenarios A-D", "status": "PASS"},
        {"check": "Five classical agents present", "status": "PASS"},
        {"check": "Scenario A -> scale_up", "status": "PASS"},
        {"check": "Scenario B -> scale_down (reflex/utility/learning)", "status": "PASS"},
        {"check": "Scenario C -> enable_autoscaling (reflex)", "status": "PASS"},
        {"check": "Edge scenario D handled", "status": "PASS"},
        {"check": "Mock tools (>=2) evidence", "status": "PASS"},
        {"check": "LLM path grounded recommendation", "status": "PASS"},
        {"check": "Learning history update", "status": "PASS"},
    ]
)
try:
    from IPython.display import display
    display(scorecard)
except Exception:
    print(scorecard.to_string(index=False))

out = {
    "classical_pivot": pivot.fillna("").to_dict(),
    "llm_agent": llm["agent"],
    "llm_recommendation": llm["recommendation"],
    "llm_tools_called": tools_called,
    "learning_update": update,
    "scorecard": scorecard.to_dict(orient="records"),
}
Path("notebooks_colab_validation_report.json").write_text(json.dumps(out, indent=2), encoding="utf-8")
print("Saved notebooks_colab_validation_report.json")
print("\nALL CHECKS PASSED")


### How to run in Google Colab

1. Upload this notebook to Colab **or** open from GitHub:
   `bits-pgaiml-aai/pcam-zg519-a1-cloud-advisor` → `notebooks/Cloud_Advisor_Colab_Validation.ipynb`
2. Runtime → Run all
3. Confirm the final cell prints **ALL CHECKS PASSED**

For local Ollama LLM demos (not required for this notebook’s PASS):
```bash
ollama serve
ollama pull qwen3:latest
streamlit run app.py
```